# TLC tip classifier (PySpark + RustFS)

Binary classification: will a trip leave a tip >= 20% of the fare?
Spark runs in **local mode** inside this notebook pod and reads the bronze
bucket from RustFS over `s3a://`. Credentials come from the injected
`AWS_*` env vars (bronze-scoped key).

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('tlc-tip')
    .master('local[*]')
    .config('spark.hadoop.fs.s3a.endpoint', os.environ['AWS_ENDPOINT_URL'])
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID'])
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')        # RustFS/MinIO
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')  # endpoint is http
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.driver.memory', '4g')
    .getOrCreate())
spark

In [ ]:
bucket = os.environ.get('BRONZE_BUCKET', 'bronze')
key = 'yellow_tripdata/yellow_tripdata_2024-01.parquet'
df = spark.read.parquet(f's3a://{bucket}/{key}')
print('rows:', df.count())
df.printSchema()

## Clean + label

- Only card payments (`payment_type = 1`): cash tips are recorded as 0, so the
  label would be wrong otherwise.
- Drop dirty rows: non-positive fares, implausible distances/durations.
- **Leakage**: do NOT use `total_amount` as a feature (it already includes the tip).

In [ ]:
clean = (df
    .withColumn('duration_min',
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime'))/60)
    .filter((F.col('payment_type') == 1)
            & (F.col('fare_amount') > 0)
            & (F.col('trip_distance').between(0.1, 100))
            & (F.col('duration_min').between(1, 180)))
    .withColumn('label', (F.col('tip_amount')/F.col('fare_amount') >= 0.20).cast('double'))
    .withColumn('hour', F.hour('tpep_pickup_datetime'))
    .withColumn('dow',  F.dayofweek('tpep_pickup_datetime'))
    .withColumn('pu', F.col('PULocationID').cast('string'))
    .withColumn('do', F.col('DOLocationID').cast('string')))

# Check class balance before trusting accuracy.
clean.groupBy('label').count().show()

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

idx = StringIndexer(inputCols=['pu','do'], outputCols=['pu_i','do_i'], handleInvalid='keep')
ohe = OneHotEncoder(inputCols=['pu_i','do_i'], outputCols=['pu_v','do_v'])
vec = VectorAssembler(
    inputCols=['trip_distance','duration_min','fare_amount','hour','dow','passenger_count','pu_v','do_v'],
    outputCol='features', handleInvalid='skip')
gbt = GBTClassifier(labelCol='label', maxIter=50, maxDepth=6)
pipeline = Pipeline(stages=[idx, ohe, vec, gbt])

train, test = clean.randomSplit([0.8, 0.2], seed=42)

## Train + track with MLflow

No MLflow server yet, so this logs to a local file store under
`~/mlruns`. When the MLflow server is deployed, point
`MLFLOW_TRACKING_URI` at it and this same code registers remotely.

In [ ]:
import mlflow, mlflow.spark
mlflow.set_experiment('tlc-tip-classifier')

with mlflow.start_run():
    model = pipeline.fit(train)
    auc = BinaryClassificationEvaluator(metricName='areaUnderROC').evaluate(model.transform(test))
    mlflow.log_param('maxIter', 50)
    mlflow.log_param('maxDepth', 6)
    mlflow.log_metric('auc', auc)
    mlflow.spark.log_model(model, 'model')
    print('AUC:', auc)

### Next steps
- Class imbalance: if skewed, use AUC-PR or a `weightCol`.
- Temporal split (train Jan-Apr, test May-Jun) instead of `randomSplit` for a realistic estimate.
- Schema drift across months: read each file, cast to a common schema, then `unionByName`.